# Extraction and structuring

Jev does not invent a value. It checks one, or it picks from values Python already found. Dates are read as separate choices. Python does the calendar math.


In [1]:
import sys
from datetime import date
from pathlib import Path
import json
import re
import statistics

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from langchain_typesafe import Choice, Noul, NoulCriteria, Score
from jev_examples.settings import ask, ask_many, draft, jev_model, openai_ready, show, typesafe_ready
from jev_examples.sample_data import (
    corpus_docs,
    customers,
    emails,
    load_json,
    lookup_order,
    open_incidents,
    order,
    products,
    read_text,
    ticket,
    tickets,
)

print("Jev model:", jev_model())
print("Jev key set:", typesafe_ready())
print("OpenAI key set:", openai_ready())


Jev model: jev-latest
Jev key set: True
OpenAI key set: True


## needs OpenAI — 18. Check an extraction field by field

A small model extracts fields. Jev asks, one field at a time, whether the value is actually in the invoice. The canned extract below puts the $12 shipping line in `amount_due` so a failed check is easy to see. With an OpenAI key, the cell uses the model's JSON instead.


In [2]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    invoice = load_json("invoices.json")
    raw = draft(
        "Extract JSON with keys invoice_number, customer_name, amount_due, payment_terms from this invoice. JSON only.\n"
        + invoice["text"]
    )
    if raw is None:
        extracted = invoice["canned_extract"]
        print("using canned extract")
    else:
        print(raw)
        text = raw if isinstance(raw, str) else str(raw)
        text = text.strip()
        if text.startswith("```"):
            text = text.strip("`")
            if text.lower().startswith("json"):
                text = text[4:].strip()
        start, end = text.find("{"), text.rfind("}")
        try:
            extracted = json.loads(text[start : end + 1])
        except json.JSONDecodeError:
            extracted = invoice["canned_extract"]
            print("reply was not JSON, using canned extract")
    questions = {
        "ok_%s" % name: Noul(
            instructions={
                "field": name,
                "extracted_value": str(value),
                "question": "Does `extracted_value` match the field as it appears in `source_text`?",
            }
        )
        for name, value in extracted.items()
    }
    response = ask({"source_text": invoice["text"]}, questions)
    show(response)
    doubtful = [name for name, answer in response.nouls.items() if answer.noul < 0.7]
    print("send back to a stronger model:", doubtful or "none")


```json
{
  "invoice_number": "INV-2044",
  "customer_name": "Acme Holdings",
  "amount_due": 240.00,
  "payment_terms": "net 30"
}
```


model: jev-1.13.0
  noul   ok_invoice_number: 0.99
  noul   ok_customer_name: 0.99
  noul   ok_amount_due: 0.92
  noul   ok_payment_terms: 0.99
send back to a stronger model: none


**What you should see.** On the canned extract, `amount_due` of 12.00 should be the doubtful field. The invoice number and customer name should pass.


## 19. Code finds the numbers, Jev picks the right one

A regex lists every amount. Jev chooses which candidate is the amount due. The stored value is always one of those spans.


In [3]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    text = load_json("invoices.json")["text"]
    candidates = re.findall(r"\$?\d[\d,]*\.\d{2}", text)
    print("candidates:", candidates)
    response = ask(
        {"text": text},
        {
            "which": Choice(
                instructions="Which candidate is the amount due in `text`?",
                criteria={item: None for item in dict.fromkeys(candidates)},
            ),
            "present": Noul(instructions="Does `text` state an amount due?"),
        },
    )
    show(response)
    if response.nouls["present"].noul < 0.5 or response.choices["which"].confidence < 0.5:
        print("route: no value")
    else:
        print("amount due:", response.choices["which"].choice)


candidates: ['$240.00', '$12.00']


model: jev-1.13.0
  noul   present: 0.99
  choice which: $240.00  (confidence 1.00)
amount due: $240.00


**What you should see.** The amount due should be `$240.00` or `240.00`, not the $12 shipping line.


## 20. Read a date as choices, compare it in Python

Jev is weak at date math. Month, day, and year are closed sets. Python builds the date and decides if it is overdue.


In [4]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    text = load_json("invoices.json")["text"]
    months = {f"{month:02d}": None for month in range(1, 13)}
    months["not_stated"] = None
    days = {str(day): None for day in range(1, 32)}
    days["not_stated"] = None
    response = ask(
        {"text": text},
        {
            "month": Choice(instructions="Which month is the due date in `text`?", criteria=months),
            "day": Choice(instructions="Which day of the month is the due date in `text`?", criteria=days),
            "year": Choice(
                instructions="Which year is the due date in `text`?",
                criteria={"2025": None, "2026": None, "2027": None, "not_stated": None},
            ),
        },
    )
    show(response)
    parts = {name: response.choices[name] for name in ("month", "day", "year")}
    if any(part.choice == "not_stated" or part.confidence < 0.5 for part in parts.values()):
        print("route: date not stated")
    else:
        due = date(int(parts["year"].choice), int(parts["month"].choice), int(parts["day"].choice))
        print("due:", due.isoformat(), "overdue:", due < date.today())


model: jev-1.13.0
  choice month: 10  (confidence 1.00)
  choice day: 15  (confidence 1.00)
  choice year: 2026  (confidence 1.00)
due: 2026-10-15 overdue: False


**What you should see.** The due date should be 2026-10-15. Whether it is overdue depends on the day you run the notebook.


## 21. Rebuild blocks from flattened lines

First ask which line breaks split a sentence. Then label each block. This stays small: six lines from the fixtures.


In [5]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    lines = load_json("content.json")["flattened_lines"]
    join_questions = {
        "joins_%s" % i: Noul(
            instructions="Does the break between lines[%s] and lines[%s] split one sentence?" % (i, i + 1)
        )
        for i in range(len(lines) - 1)
    }
    joined = ask({"lines": lines}, join_questions)
    show(joined)
    blocks = [lines[0]]
    for i in range(len(lines) - 1):
        if joined.nouls["joins_%s" % i].noul > 0.5:
            blocks[-1] = blocks[-1] + " " + lines[i + 1]
        else:
            blocks.append(lines[i + 1])
    print("blocks:", blocks)
    kind_questions = {
        "kind_%s" % i: Choice(
            instructions="What kind of block is blocks[%s]?" % i,
            criteria={"heading": None, "paragraph": None, "list_item": None, "code": None},
        )
        for i in range(len(blocks))
    }
    kinds = ask({"blocks": blocks}, kind_questions)
    show(kinds)
    for i, block in enumerate(blocks):
        print(kinds.choices["kind_%s" % i].choice, ":", block)


model: jev-1.13.0
  noul   joins_0: 0.23
  noul   joins_1: 0.93
  noul   joins_2: 0.12
  noul   joins_3: 0.17
  noul   joins_4: 0.10
blocks: ['Refund policy', 'Northwind accepts returns for 30 days after purchase.', '- Duplicate charges are refundable', '```', 'order_id = A-104']


model: jev-1.13.0
  choice kind_0: heading  (confidence 0.85)
  choice kind_1: list_item  (confidence 0.71)
  choice kind_2: code  (confidence 0.46)
  choice kind_3: code  (confidence 0.87)
  choice kind_4: code  (confidence 0.87)
heading : Refund policy
list_item : Northwind accepts returns for 30 days after purchase.
code : - Duplicate charges are refundable
code : ```
code : order_id = A-104


**What you should see.** The '30' / 'days after purchase' pair should join into one sentence. The heading, list item, and code line should stay separate.


## 22. Are these two records the same shopper?

The score levels are the actions: leave them, send to a person, or merge. No extra threshold to invent.


In [6]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    people = {row["id"]: row for row in customers()}
    pairs = [("C08", "C09"), ("C01", "C12")]
    questions = {
        "same_entity": Score(
            instructions="Do `record_a` and `record_b` describe the same real-world person or company?",
            criteria=[
                "Different: leave unlinked",
                "Possibly the same: send to a person",
                "Clearly the same: merge",
            ],
        )
    }
    requests = [
        {"state": {"record_a": people[a], "record_b": people[b]}, "questions": questions}
        for a, b in pairs
    ]
    for pair, response in zip(pairs, ask_many(requests)):
        show(response)
        score = response.scores["same_entity"].score
        if score > 1.5:
            route = "merge"
        elif score > 0.5:
            route = "curator"
        else:
            route = "unlinked"
        print(pair, "->", route)


model: jev-1.13.0
  score  same_entity: 1.92  ~ Clearly the same: merge
('C08', 'C09') -> merge
model: jev-1.13.0
  score  same_entity: 0.00  ~ Different: leave unlinked
('C01', 'C12') -> unlinked


**What you should see.** Jonah Brooks and J. Brooks share a street and should at least go to a curator, often merge. Maya Chen and Acme Holdings should stay unlinked.
